# Data Leakage in den Ranglistenpositionen und Team Ranking

- Rangliste = Zustand Ende der Saison 
- somit Grand Tours bereits mit eingepreist
- Lösung: Den Vorjahreswert als Ausgangspunkt in die Modelle bringen

- dazu Scraper 08...py, um die Werte für 2004 zu bekommen
    - joinen mit unserem Datensatz inkl lagging -> Werte 2006 sind bspw. Startwert für 2007
    - dazu neue lag_ spalten erstellen für Sauberkeit
    
- Team Index neu berechnen (vgl. Notebook 07-11)
- Test Missing Values

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

In [16]:
pickle_path = '../../data/processed/25_cleaned_master_data.pkl'

df = pd.read_pickle(pickle_path)

In [3]:
print(df.columns.tolist())

['meta_race', 'meta_year', 'meta_url', 'rank', 'meta_rider_url', 'height', 'meta_name', 'meta_nationality', 'weight', 'meta_url_name', 'meta_departure', 'meta_arrival', 'distance', 'vertical_meters', 'one_day_races', 'gc', 'time_trial', 'sprint', 'climber', 'hills', 'stage_nr', 'meta_date', 'meta_departure_lat', 'meta_departure_lon', 'meta_arrival_lat', 'meta_arrival_lon', 'rider_points_season', 'rider_rank_season', 'meta_current_team', 'team_tier', 'age_at_race', 'rider_bmi', 'race_competitiveness_median', 'team_power_index', 'wind_stability_index', 'weather_temp_mean', 'weather_temp_trend', 'weather_rain_prob_mean', 'weather_precipitation_mean', 'weather_humidity_mean', 'won_how_cat', 'gradient_final_km']


# Daten "Laggen" 'rider_points_season', 'rider_rank_season' 
- damit die Daten aus 2005 nun als Basis für 2006 dienen bspw.

In [ ]:
# Erzeugung Hilfsdatensatz
# alle Gruppiert nach dem Jahr die Punkte und die Ranks pro Saison extrahieren
rider_year_df = (
    df[
        [
            "meta_rider_url",
            "meta_year",
            "rider_points_season",
            "rider_rank_season"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["meta_rider_url", "meta_year"]
    )
)

print(rider_year_df.shape)
rider_year_df.head()

(9436, 4)


,meta_rider_url,meta_year,rider_points_season,rider_rank_season
169340,aaron-gate,2017,69,791
134306,aaron-kemps,2006,200,296
67448,aart-vierhouten,2005,295,198
187652,abel-balderstone-roumens,2023,107,529
193593,abel-balderstone-roumens,2025,251,258


In [29]:
rider_year_df[
    rider_year_df["meta_rider_url"] == "aaron-kemps"
].sort_values("meta_year")

,meta_rider_url,meta_year,rider_points_season,rider_rank_season
134306,aaron-kemps,2006,200,296


In [25]:
# Innerhalb des Hilfsdatensatzes die beiden lag-features erzeugen mittels shift()

rider_year_df["lag_rider_points_season"] = (
    rider_year_df
    .groupby("meta_rider_url")["rider_points_season"]
    .shift(1)
)

rider_year_df["lag_rider_rank_season"] = (
    rider_year_df
    .groupby("meta_rider_url")["rider_rank_season"]
    .shift(1)
)

rider_year_df.head()

,meta_rider_url,meta_year,rider_points_season,rider_rank_season,lag_rider_points_season,lag_rider_rank_season
169340,aaron-gate,2017,69,791,NaN,NaN
134306,aaron-kemps,2006,200,296,NaN,NaN
67448,aart-vierhouten,2005,295,198,NaN,NaN
187652,abel-balderstone-roumens,2023,107,529,NaN,NaN
193593,abel-balderstone-roumens,2025,251,258,107.0,529.0


In [26]:
rider_year_df[
    rider_year_df["meta_rider_url"] == "aaron-kemps"
].sort_values("meta_year")

,meta_rider_url,meta_year,rider_points_season,rider_rank_season,lag_rider_points_season,lag_rider_rank_season
134306,aaron-kemps,2006,200,296,NaN,NaN


# Joinen der 2004er Daten mit den Rennen aus dem Jahr 2005

In [ ]:
# Die Daten aus dem Scraper 08 laden
rank_df = pd.read_json(
    "../../data/raw/rider_rank_points_2004.jsonl",
    lines=True
)

rank_df.head()

,url_name,season,points,rank
0,tom-boonen,2004,1593,11
1,thor-hushovd,2004,1224,17
2,robbie-mcewen,2004,1678,8
3,stuart-o-grady,2004,1866,5
4,luciano-andre-pagliarini-mendonca,2004,179,335


In [ ]:


rank_df = rank_df.rename(
    columns={
        "points": "new_rider_points_season",
        "rank": "new_rider_rank_season"
    }
)[
    [
        "url_name",
        "new_rider_points_season",
        "new_rider_rank_season"
    ]
]

# Join
df = df.merge(
    rank_df,
    how="left",
    left_on="meta_rider_url",
    right_on="url_name"
)

# Nur Jahr 2005 aktualisieren
mask = (
    (df["meta_year"] == 2005)
    & (df["new_rider_points_season"].notna())
)

df.loc[mask, "rider_points_season"] = df.loc[
    mask, "new_rider_points_season"
]

df.loc[mask, "rider_rank_season"] = df.loc[
    mask, "new_rider_rank_season"
]

# Hilfsspalten entfernen
df.drop(
    columns=[
        "url_name",
        "new_rider_points_season",
        "new_rider_rank_season"
    ],
    inplace=True
)

In [9]:
print(mask.sum())

8633


In [ ]:
# Kurzer Test

df.loc[
    (df["meta_rider_url"] == "jan-ullrich")
    & (df["meta_year"] == 2005),
    [
        "meta_year",
        "meta_rider_url",
        "rider_points_season",
        "rider_rank_season"
    ]
].sort_values("meta_year")

,meta_year,meta_rider_url,rider_points_season,rider_rank_season
18,2005,jan-ullrich,1378,14
222,2005,jan-ullrich,1378,14
407,2005,jan-ullrich,1378,14
584,2005,jan-ullrich,1378,14
797,2005,jan-ullrich,1378,14
943,2005,jan-ullrich,1378,14
1146,2005,jan-ullrich,1378,14
1305,2005,jan-ullrich,1378,14
1480,2005,jan-ullrich,1378,14
1682,2005,jan-ullrich,1378,14


# Team Index neu berechnen

# NA Test

# Checkpointing